# GeoCLIP — Training, Evaluation & Interpretability

Run on a **local Jupyter server** (or Google Colab with a GPU runtime).

| Section | What it does |
|---|---|
| 1 | Environment setup |
| 2 | Configuration |
| 3 | Data & model |
| 4 | Training |
| 5 | Evaluation (GCD + recall@km) |
| 6 | Grad-CAM |
| 7 | Attention Rollout |
| 8 | Integrated Gradients |
| 9 | Layer-wise Grad-CAM |
| 10 | Per-head Attention |
| 11 | World Map Similarity |
| 12 | All-methods comparison |
| 13 | Good vs. bad predictions |
| 14 | Köppen-Geiger climate analysis |
| 15 | Resume training |
| 16 | Training curves |
| 17 | Error distribution |
| 18 | Embedding space (t-SNE) |
| 19 | Top-K gallery retrieval |
| 20 | Similarity–error calibration |
| 21 | Per-Köppen-zone performance |
| 22 | Ablation study |

## 1. Setup

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — switch runtime to T4 GPU.')

In [ ]:
import os

# ── Configure your local paths ────────────────────────────────────────────────
# Set BASE_DIR to the folder where downloads and outputs should be stored.
# Sub-folders will be created automatically.
BASE_DIR = '/your/local/path/geoclip'   # <-- change this

OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
HF_HOME    = os.path.join(BASE_DIR, 'hf_home')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(HF_HOME,    exist_ok=True)

# Redirect ALL HuggingFace activity (datasets, hub, tokenizer cache) to HF_HOME
# before any HF library is imported.
os.environ['HF_HOME'] = HF_HOME

# Alias used throughout the notebook for output paths (checkpoints, plots, etc.)
DRIVE_DIR = OUTPUT_DIR

print(f'Output dir : {OUTPUT_DIR}')
print(f'HF home    : {HF_HOME}')

In [ ]:
import sys, os

# Locate the repo root: if the notebook is run from notebooks/, go up one level;
# otherwise assume CWD is already the repo root.
_cwd = os.getcwd()
REPO_DIR = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'notebooks' else _cwd

sys.path.insert(0, REPO_DIR)
print(f'Repo dir: {REPO_DIR}')

In [ ]:
# Install dependencies if not already present (safe to re-run).
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'transformers',
    'datasets>=2.14.0,<3.0.0', 'webdataset>=0.2.86',
    'pyyaml', 'tqdm', 'matplotlib', 'numpy', 'scikit-learn',
])

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from collections import Counter
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}', torch.cuda.get_device_name(0) if device == 'cuda' else '')

## 2. Configuration

Edit the YAML below. Colab-friendly defaults train in ~20 min on a T4.
See the README for a description of every parameter.

In [ ]:
CONFIG_YAML = f"""
model:
  clip_backbone: ViT-B/32   # ViT-B/32 (fast) or ViT-B/16 (better accuracy)
  freeze_layers: 11          # freeze all transformer blocks except the last one
  rff_num_scales: 4
  rff_dim: 64
  mlp_hidden: 256
  embedding_dim: 512

data:
  subset_size: null          # raise to 50000 for a real run
  num_shards: 10             # limit number of wds shards (None = all)
  streaming: false
  num_workers: 2
  pin_memory: true

gallery:
  size: 1000
  strategy: train_sample
  cache_path: {OUTPUT_DIR}/gallery.pt

training:
  batch_size: 64
  epochs: 5
  warmup_epochs: 1
  lr_clip: 1.0e-5
  lr_gps: 1.0e-4
  lr_temp: 1.0e-3
  weight_decay: 0.1
  amp: true
  log_every: 10
  eval_every: 1
  checkpoint_dir: {OUTPUT_DIR}/checkpoints
  hard_neg_swap_prob: 0.5
  hard_neg_min_dist_km: 500.0
  lambda_attn: 0.01
  attn_reg_every: 4

evaluation:
  thresholds_km: [1, 25, 200, 750, 2500]
"""

CONFIG_PATH = os.path.join(OUTPUT_DIR, 'config.yaml')
with open(CONFIG_PATH, 'w') as f:
    f.write(CONFIG_YAML)

from geoclip.utils.config import load_config
cfg = load_config(CONFIG_PATH)
print('Loaded config. Backbone:', cfg.model.clip_backbone, '| Epochs:', cfg.training.epochs)
print('Checkpoint dir:', cfg.training.checkpoint_dir)

## 3. Data & model

In [ ]:
from geoclip.data.dataset import OSV5MDataset
from geoclip.data.shard_dataset import ShardedOSV5MDataset, StreamingOSV5MDataset
from geoclip.data.transforms import get_train_transform, get_eval_transform

# ── Dataset mode ──────────────────────────────────────────────────────────────
# "hf"        → standard HuggingFace download (osv5m/osv5m). Downloads the full
#               split once and caches it; uses subset_size to cap the number of
#               samples loaded into RAM. Best for local runs with enough disk.
# "streaming" → fetches samples lazily from the wds Hub; no pre-download.
#               Best for quick experiments when disk space is limited.
# "sharded"   → downloads one wds shard (~670 MB) at a time with background
#               prefetch; good for long local/server runs.
# "subset"    → same as "hf" but always streams and caps at subset_size.
DATASET_MODE = "hf"   # "hf" | "streaming" | "sharded" | "subset"
# ─────────────────────────────────────────────────────────────────────────────

if DATASET_MODE == "hf":
    train_dataset = OSV5MDataset(
        split="train",
        subset_size=cfg.data.subset_size,
        transform=get_train_transform(),
        cache_dir=HF_HOME,
    )
    train_loader = DataLoader(train_dataset, batch_size=cfg.training.batch_size,
                              shuffle=True, num_workers=cfg.data.num_workers,
                              pin_memory=cfg.data.pin_memory)
    print(f"HF mode — {len(train_dataset)} samples, cached in {HF_HOME}")

elif DATASET_MODE == "streaming":
    train_dataset = StreamingOSV5MDataset(
        split="train",
        num_shards=cfg.data.get("num_shards"),
        transform=get_train_transform(),
        shuffle_buffer=4096,
        hf_home=HF_HOME,
    )
    # webdataset distributes shards across workers — num_workers > 0 is fine.
    train_loader = DataLoader(train_dataset, batch_size=cfg.training.batch_size,
                              shuffle=False, num_workers=cfg.data.num_workers,
                              prefetch_factor=2 if cfg.data.num_workers > 0 else None,
                              pin_memory=cfg.data.pin_memory)
    print("Streaming mode — webdataset, no pre-download")

elif DATASET_MODE == "sharded":
    train_dataset = ShardedOSV5MDataset(
        split="train",
        num_shards=cfg.data.get("num_shards"),
        transform=get_train_transform(),
        shards_per_step=1,
        hf_home=HF_HOME,
    )
    train_loader = DataLoader(train_dataset, batch_size=cfg.training.batch_size,
                              shuffle=True, num_workers=cfg.data.num_workers,
                              pin_memory=cfg.data.pin_memory)
    print(f"Sharded mode — disk ~{train_dataset.shards_per_step * 670} MB, "
          f"rotating every epoch")

else:  # subset — streams from HF and caps at subset_size
    train_dataset = OSV5MDataset(
        split="train",
        subset_size=cfg.data.subset_size,
        transform=get_train_transform(),
        cache_dir=HF_HOME,
    )
    train_loader = DataLoader(train_dataset, batch_size=cfg.training.batch_size,
                              shuffle=True, num_workers=cfg.data.num_workers,
                              pin_memory=cfg.data.pin_memory)
    print(f"Subset mode — {len(train_dataset)} samples in RAM")

# Validation set — always use the standard HF loader (small enough).
val_raw = OSV5MDataset(
    split="test",
    subset_size=None,
    transform=get_eval_transform(),
    cache_dir=HF_HOME,
)
val_size = min(max(500, cfg.data.subset_size // 10), len(val_raw))
val_dataset = torch.utils.data.Subset(val_raw, list(range(val_size)))
val_loader  = DataLoader(val_dataset, batch_size=64,
                         shuffle=False, num_workers=cfg.data.num_workers,
                         pin_memory=cfg.data.pin_memory)

if DATASET_MODE not in ("streaming",):
    print(f"Train: {len(train_dataset)} samples | Test: {val_size} samples")
else:
    print(f"Test: {val_size} samples")

In [ ]:
from geoclip.models.geoclip import GeoCLIP
from geoclip.data.gallery import load_or_build_gallery

model = GeoCLIP(
    clip_model_name=cfg.model.clip_backbone,
    freeze_layers=cfg.model.freeze_layers,
    rff_num_scales=cfg.model.rff_num_scales,
    rff_dim=cfg.model.rff_dim,
    mlp_hidden=cfg.model.mlp_hidden,
    embedding_dim=cfg.model.embedding_dim,
)

gallery_coords = load_or_build_gallery(
    strategy=cfg.gallery.strategy,
    size=cfg.gallery.size,
    cache_path=cfg.gallery.cache_path,
    dataset=train_dataset,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parameters: {trainable:,} trainable / {total:,} total')
print(f'Gallery: {len(gallery_coords)} GPS points')

## 4. Training

Checkpoints are written to `OUTPUT_DIR/checkpoints/`. Set `resume_path` in
section 15 to continue from a previous run.

In [ ]:
from geoclip.training.trainer import Trainer

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    gallery_coords=gallery_coords,
    cfg=cfg,
    device=device,
    resume_path=None,  # set to a Drive path to resume a previous run
)
trainer.train()

## 5. Evaluation

Gallery-based retrieval: for each image the predicted location is the gallery
point with the highest cosine similarity to the image embedding.
Reported metrics: mean / median great-circle distance (km) and recall@{1,25,200,750,2500} km.

In [ ]:
from geoclip.utils.checkpoint import load_checkpoint
from geoclip.training.evaluator import evaluate
from geoclip.data.gallery import compute_gallery_embeddings
from geoclip.utils.geo_math import haversine_distance

CHECKPOINT = f'{DRIVE_DIR}/checkpoints/best.pt'
if os.path.exists(CHECKPOINT):
    load_checkpoint(CHECKPOINT, model, device=device)
    print('Loaded best checkpoint.')
else:
    print('No checkpoint on Drive — using current weights.')

model = model.to(device).eval()

In [ ]:
metrics = evaluate(model, val_loader, gallery_coords,
                   device=device, thresholds_km=cfg.evaluation.thresholds_km)

print('=== Results ===')
print(f"Mean GCD:   {metrics['mean_gcd_km']:>8.1f} km")
print(f"Median GCD: {metrics['median_gcd_km']:>8.1f} km")
for k, v in metrics.items():
    if k.startswith('recall'):
        print(f"  {k}: {v*100:.1f}%")

---
## Interpretability setup

The cells below run all six interpretability methods on the same batch of images.
Run this cell first — it loads the images and pre-computes gallery embeddings
used by every subsequent section.

In [ ]:
# Sample a fixed batch for all interpretability methods
torch.manual_seed(0)
interp_loader = DataLoader(val_dataset, batch_size=8, shuffle=True)
images, true_coords = next(iter(interp_loader))

gallery_embs = compute_gallery_embeddings(model, gallery_coords, device=device)

with torch.no_grad():
    img_embs = model.encode_image(images.to(device))
    best_idx = (img_embs @ gallery_embs.to(device).T).argmax(dim=-1)
pred_coords = gallery_coords[best_idx.cpu()]
distances   = haversine_distance(pred_coords, true_coords)

N = 4  # number of samples to display in each figure
print(f'Median error on this batch: {distances.median():.0f} km')

In [ ]:
# Shared helpers used by all visualization cells
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073])
CLIP_STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711])

def denormalize(t):
    img = t.cpu() * CLIP_STD[:, None, None] + CLIP_MEAN[:, None, None]
    return (img.clamp(0, 1).permute(1, 2, 0).numpy() * 255).astype(np.uint8)

def overlay(img_np, hmap, alpha=0.5):
    colored = (cm.jet(hmap)[:, :, :3] * 255).astype(np.uint8)
    return (alpha * colored + (1 - alpha) * img_np).astype(np.uint8)

def show_grid(rows, col_titles, suptitle='', save_as=None):
    n_rows, n_cols = len(rows), len(rows[0])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.5 * n_cols, 3.5 * n_rows))
    if n_rows == 1: axes = [axes]
    for r, row in enumerate(rows):
        for c, img in enumerate(row):
            axes[r][c].imshow(img)
            axes[r][c].axis('off')
            if r == 0:
                axes[r][c].set_title(col_titles[c], fontsize=9, fontweight='bold')
    if suptitle: plt.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    if save_as: plt.savefig(f'{DRIVE_DIR}/{save_as}', dpi=150, bbox_inches='tight')
    plt.show()

img_nps = [denormalize(images[i]) for i in range(N)]

## 6. Grad-CAM

Back-propagates the image–GPS cosine similarity through the last transformer block.
Patch activations are weighted by their mean gradient → highlights which patches
most increased the similarity score.

In [ ]:
from geoclip.interpretability.gradcam import gradcam_context

with gradcam_context(model) as gc:
    gcam_maps = gc.compute(images[:N], true_coords[:N])

rows = [[img_nps[i], overlay(img_nps[i], gcam_maps[i].numpy())] for i in range(N)]
show_grid(rows, ['Input', 'Grad-CAM'], 'Grad-CAM', save_as='gradcam.png')

## 7. Attention Rollout

Accumulates attention across all transformer layers: `R = (Â₁₂+I) ⊗ … ⊗ (Â₁+I)`.
The CLS row of R shows the effective attention each patch received end-to-end,
without the compounding noise of a single layer.

In [ ]:
from geoclip.interpretability.attention_rollout import attention_rollout_context

with attention_rollout_context(model, head_fusion='mean', discard_ratio=0.1) as ar:
    rollout_maps = ar.compute(images[:N])

rows = [[img_nps[i], overlay(img_nps[i], rollout_maps[i].numpy())] for i in range(N)]
show_grid(rows, ['Input', 'Attention Rollout'], 'Attention Rollout', save_as='rollout.png')

## 8. Integrated Gradients

Averages gradients along the path from a grey baseline to the real image:

$$\text{IG}(x) = (x - x') \times \frac{1}{N}\sum_{k=1}^{N} \frac{\partial f}{\partial x}\!\left(x' + \tfrac{k}{N}(x-x')\right)$$

**Completeness:** attributions sum to exactly `f(x) − f(baseline)`, making them verifiable.

In [ ]:
from geoclip.interpretability.integrated_gradients import IntegratedGradients

ig = IntegratedGradients(model, n_steps=50)
ig_maps = ig.compute(images[:N], true_coords[:N])

rows = [[img_nps[i], overlay(img_nps[i], ig_maps[i].numpy())] for i in range(N)]
show_grid(rows, ['Input', 'Integrated Gradients'], 'Integrated Gradients', save_as='ig.png')

In [ ]:
# Completeness check: sum of signed IG attributions ≈ f(x) − f(baseline)
baseline = torch.zeros_like(images[:1])
with torch.no_grad():
    gps_emb      = model.encode_gps(true_coords[:1].to(device))
    f_x          = (model.encode_image(images[:1].to(device)) * gps_emb).sum().item()
    f_base       = (model.encode_image(baseline.to(device))   * gps_emb).sum().item()

accum = torch.zeros_like(images[:1])
for step in range(1, 51):
    alpha  = step / 50
    interp = (baseline + alpha * (images[:1] - baseline)).detach().requires_grad_(True)
    score  = (model.encode_image(interp.to(device)) * gps_emb.detach()).sum()
    score.backward()
    accum += interp.grad.detach().cpu()
signed_sum = ((images[:1] - baseline) * (accum / 50)).sum().item()

print(f'f(x) - f(baseline) = {f_x - f_base:.6f}')
print(f'Sum of IG          = {signed_sum:.6f}')
print(f'Completeness error : {abs((f_x - f_base) - signed_sum):.2e}  (should be ~0)')

## 9. Layer-wise Grad-CAM

Runs Grad-CAM at blocks 2, 5, 8, 11 to show how geographic abstraction builds with depth.
Early blocks respond to texture; later blocks to higher-level geographic structure.

In [ ]:
from geoclip.interpretability.gradcam import gradcam_layerwise

layer_indices = (2, 5, 8, 11)
layerwise = gradcam_layerwise(model, images[:N], true_coords[:N], layer_indices=layer_indices)

rows = [
    [img_nps[i]] + [overlay(img_nps[i], layerwise[idx][i].numpy()) for idx in layer_indices]
    for i in range(N)
]
col_titles = ['Input'] + [f'Block {idx}' for idx in layer_indices]
show_grid(rows, col_titles, 'Layer-wise Grad-CAM', save_as='layerwise_gradcam.png')

## 10. Per-head Attention

Shows the CLS→patches attention map for each of the 12 heads in the last layer.
Heads often specialize: some attend to local texture, others to global layout or signage.

In [ ]:
from geoclip.interpretability.attention_rollout import PerHeadAttention

pha = PerHeadAttention(model, layer_idx=-1)
head_maps = pha.compute(images[:N])  # [N, num_heads, H, W]
num_heads = head_maps.shape[1]

sample_idx = 0  # change to inspect a different image
img_np = img_nps[sample_idx]
cols = 4
rows_h = (num_heads + cols - 1) // cols

fig, axes = plt.subplots(rows_h, cols * 2, figsize=(cols * 4, rows_h * 2.5))
for h in range(num_heads):
    r, c = divmod(h, cols)
    hmap = head_maps[sample_idx, h].numpy()
    axes[r][c * 2].imshow(img_np)
    axes[r][c * 2].axis('off')
    axes[r][c * 2 + 1].imshow(overlay(img_np, hmap))
    axes[r][c * 2 + 1].set_title(f'Head {h}', fontsize=7)
    axes[r][c * 2 + 1].axis('off')
for h in range(num_heads, rows_h * cols):
    r, c = divmod(h, cols)
    axes[r][c * 2].set_visible(False)
    axes[r][c * 2 + 1].set_visible(False)

plt.suptitle(f'Per-head Attention (last layer) — sample {sample_idx}', fontsize=11)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/per_head_attention.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. World Map Similarity Heatmap

Plots the full similarity distribution over the gallery rather than just the argmax.
A sharp peak = confident prediction. Mass spread across two continents = the model
is responding to a spurious visual cue shared by both regions.

In [ ]:
from geoclip.interpretability.world_map import plot_similarity_map

with torch.no_grad():
    sims_all = (model.encode_image(images[:N].to(device)) @ gallery_embs.to(device).T).cpu()

fig, axes = plt.subplots(1, N, figsize=(6 * N, 3))
if N == 1: axes = [axes]
for i in range(N):
    plot_similarity_map(
        gallery_coords, sims_all[i],
        true_coords=true_coords[i], pred_coords=pred_coords[i],
        title=f'Sample {i} — {distances[i]:.0f} km error',
        ax=axes[i],
    )
plt.suptitle('Geographic Belief Distribution  (★ = true, ✕ = pred)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/worldmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. All-methods comparison

All five spatial methods on the same image. When Grad-CAM, Rollout, and IG all
highlight the same region and that region contains a recognizable geographic cue,
the model is behaving as intended.

In [ ]:
i = 0  # change to inspect a different sample
img_np = img_nps[i]

with gradcam_context(model) as gc:
    gcam = gc.compute(images[i:i+1], true_coords[i:i+1])[0].numpy()
with attention_rollout_context(model, head_fusion='mean', discard_ratio=0.1) as ar:
    roll = ar.compute(images[i:i+1])[0].numpy()
ig_map = IntegratedGradients(model, n_steps=50).compute(images[i:i+1], true_coords[i:i+1])[0].numpy()
lw     = gradcam_layerwise(model, images[i:i+1], true_coords[i:i+1], layer_indices=(2, 5, 8, 11))
ph_mean = PerHeadAttention(model, layer_idx=-1).compute(images[i:i+1])[0].mean(dim=0).numpy()

methods = {'Grad-CAM': gcam, 'Rollout': roll, 'Integ. Grad': ig_map,
           'GC Block 11': lw[11][0].numpy(), 'Heads (avg)': ph_mean}

fig, axes = plt.subplots(1, 1 + len(methods), figsize=(3.5 * (1 + len(methods)), 3.5))
axes[0].imshow(img_np)
axes[0].set_title(f'Input\nTrue: ({true_coords[i,0]:.1f}, {true_coords[i,1]:.1f})\n'
                  f'Pred: {distances[i]:.0f} km away', fontsize=7)
axes[0].axis('off')
for ax, (name, hmap) in zip(axes[1:], methods.items()):
    ax.imshow(overlay(img_np, hmap))
    ax.set_title(name, fontsize=8, fontweight='bold')
    ax.axis('off')
plt.suptitle('All methods — sample comparison', fontsize=11)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/all_methods.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Good vs. bad predictions

Compare what the model attends to for well-localized images vs. failures.
Failures highlighting backgrounds or generic textures indicate spurious correlations.

In [ ]:
good_mask = distances < 200
bad_mask  = distances > 750
print(f'Good (<200 km): {good_mask.sum().item()}  |  Bad (>750 km): {bad_mask.sum().item()}')

if good_mask.any() and bad_mask.any():
    gi = good_mask.nonzero()[0].item()
    bi = bad_mask.nonzero()[0].item()
    pair_images = torch.stack([images[gi], images[bi]])
    pair_coords = torch.stack([true_coords[gi], true_coords[bi]])

    with gradcam_context(model) as gc:
        gcam_pair = gc.compute(pair_images, pair_coords)
    ig_pair = IntegratedGradients(model, n_steps=50).compute(pair_images, pair_coords)

    fig, axes = plt.subplots(2, 3, figsize=(10, 7))
    for row, (img_t, gc_map, ig_map, label, color) in enumerate(zip(
        pair_images, gcam_pair, ig_pair,
        [f'Good — {distances[gi]:.0f} km', f'Bad — {distances[bi]:.0f} km'],
        ['green', 'red'],
    )):
        p = denormalize(img_t)
        axes[row][0].imshow(p);            axes[row][0].set_title(label, fontsize=9, color=color); axes[row][0].axis('off')
        axes[row][1].imshow(overlay(p, gc_map.numpy())); axes[row][1].set_title('Grad-CAM', fontsize=9); axes[row][1].axis('off')
        axes[row][2].imshow(overlay(p, ig_map.numpy())); axes[row][2].set_title('Integrated Gradients', fontsize=9); axes[row][2].axis('off')
    plt.suptitle('Good vs. Bad Predictions', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{DRIVE_DIR}/good_vs_bad.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Not enough contrasting samples — increase batch_size in the interpretability setup cell.')

## 14. Köppen-Geiger Climate Analysis

Maps each true and predicted location to its Köppen climate class and classifies
errors as **exact** (same major group), **adjacent** (neighboring groups, e.g. C→D),
or **distant** (different climate regime — likely a spurious visual cue).

The 0.5° resolution raster is downloaded once (~0.5 MB) and cached.

In [ ]:
from geoclip.utils.koppen import (
    KoppenClassifier, classify_error,
    KOPPEN_GROUPS, GROUP_COLORS,
)

kc = KoppenClassifier()

true_np = true_coords.numpy()
pred_np = pred_coords.numpy()
true_info = kc.classify_batch(true_np[:, 0], true_np[:, 1])
pred_info = kc.classify_batch(pred_np[:, 0], pred_np[:, 1])
coherence = [classify_error(tg, pg)
             for tg, pg in zip(true_info['groups'], pred_info['groups'])]

print(f"{'#':>3}  {'Dist km':>8}  {'True':>6}  {'Pred':>6}  Coherence")
print('-' * 44)
for i, (d, tc, pc, coh) in enumerate(zip(
    distances.tolist(), true_info['classes'], pred_info['classes'], coherence
)):
    print(f'{i:>3}  {d:>8.0f}  {tc:>6}  {pc:>6}  {coh}')

In [ ]:
# Coherence pie + major-group confusion matrix
counts = Counter(coherence)
labels_order = ['exact', 'adjacent', 'distant', 'ocean']
pie_colors   = ['#2ecc71', '#f39c12', '#e74c3c', '#95a5a6']
active = [(l, counts.get(l, 0), c) for l, c in zip(labels_order, pie_colors) if counts.get(l, 0) > 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].pie([s for _, s, _ in active], labels=[f'{l}\n({s})' for l, s, _ in active],
            colors=[c for _, _, c in active], autopct='%1.0f%%', startangle=140,
            textprops={'fontsize': 9})
axes[0].set_title('Error coherence with Köppen zones', fontsize=11)

groups_order = ['A', 'B', 'C', 'D', 'E', '?']
n_g = len(groups_order)
conf = np.zeros((n_g, n_g), dtype=int)
g2i  = {g: i for i, g in enumerate(groups_order)}
for tg, pg in zip(true_info['groups'], pred_info['groups']):
    conf[g2i[tg], g2i[pg]] += 1

im = axes[1].imshow(conf, cmap='Blues')
axes[1].set_xticks(range(n_g)); axes[1].set_xticklabels(groups_order)
axes[1].set_yticks(range(n_g)); axes[1].set_yticklabels(groups_order)
axes[1].set_xlabel('Predicted group'); axes[1].set_ylabel('True group')
axes[1].set_title('Confusion matrix — major climate groups', fontsize=11)
for r in range(n_g):
    for c in range(n_g):
        if conf[r, c] > 0:
            axes[1].text(c, r, str(conf[r, c]), ha='center', va='center', fontsize=11,
                         color='white' if conf[r, c] > conf.max() * 0.5 else 'black')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
patches = [mpatches.Patch(color=GROUP_COLORS.get(g, '#fff'),
                           label=f'{g} — {KOPPEN_GROUPS.get(g, "Ocean")}') for g in groups_order if g != '?']
axes[1].legend(handles=patches, fontsize=7, bbox_to_anchor=(1.25, 1), title='Group')
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/koppen_coherence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# World map coloured by Köppen group; point size ∝ similarity to query
with torch.no_grad():
    sims_k = (model.encode_image(images[:N].to(device)) @ gallery_embs.to(device).T).cpu()

gallery_np = gallery_coords.numpy()
gal_info   = kc.classify_batch(gallery_np[:, 0], gallery_np[:, 1])
gal_colors = [GROUP_COLORS.get(g, '#aaaaaa') for g in gal_info['groups']]

fig, axes = plt.subplots(1, N, figsize=(7 * N, 3.5))
if N == 1: axes = [axes]
for i, ax in enumerate(axes):
    sims_i = sims_k[i].numpy()
    norm_s = (sims_i - sims_i.min()) / (sims_i.max() - sims_i.min() + 1e-8)
    ax.set_facecolor('#d0e8f0'); ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
    ax.set_xticks(range(-180, 181, 60)); ax.set_yticks(range(-90, 91, 30))
    ax.tick_params(labelsize=5); ax.grid(color='white', linewidth=0.3, alpha=0.5)
    ax.scatter(gallery_np[:, 1], gallery_np[:, 0], c=gal_colors,
               s=2 + 14 * norm_s, alpha=0.75, linewidths=0, zorder=2)
    tc, pc = true_coords[i].numpy(), pred_coords[i].numpy()
    ax.scatter(tc[1], tc[0], marker='*', s=220, color='lime', edgecolors='black',
               linewidths=0.5, zorder=6, label=f"True: {true_info['classes'][i]}")
    ax.scatter(pc[1], pc[0], marker='X', s=150, color='red', edgecolors='black',
               linewidths=0.5, zorder=6, label=f"Pred: {pred_info['classes'][i]}")
    ax.set_title(f'Sample {i}  |  {distances[i]:.0f} km  |  {coherence[i]}', fontsize=8)
    ax.legend(fontsize=6, loc='lower left')
patches = [mpatches.Patch(color=GROUP_COLORS[g], label=f'{g} {KOPPEN_GROUPS[g]}')
           for g in ['A', 'B', 'C', 'D', 'E']]
axes[-1].legend(handles=patches, loc='lower right', fontsize=6, title='Köppen group', title_fontsize=7)
plt.suptitle('Gallery by Köppen zone  (size ∝ similarity)  ★ true  ✕ pred', fontsize=10)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/koppen_worldmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Error coherence by distance bucket
buckets = [
    ('< 100 km',    lambda d: d < 100),
    ('100–500 km',  lambda d: 100 <= d < 500),
    ('500–2000 km', lambda d: 500 <= d < 2000),
    ('> 2000 km',   lambda d: d >= 2000),
]
coh_types  = ['exact', 'adjacent', 'distant', 'ocean']
coh_colors = ['#2ecc71', '#f39c12', '#e74c3c', '#95a5a6']
dist_list  = distances.tolist()

table = {}
for label, fn in buckets:
    idx = [j for j, d in enumerate(dist_list) if fn(d)]
    cnt = Counter(coherence[j] for j in idx)
    table[label] = [cnt.get(ct, 0) for ct in coh_types]

bucket_labels = list(table.keys())
data = np.array(list(table.values()))

fig, ax = plt.subplots(figsize=(9, 4))
bottoms = np.zeros(len(bucket_labels))
for ci, (ct, col) in enumerate(zip(coh_types, coh_colors)):
    vals = data[:, ci]
    ax.bar(bucket_labels, vals, bottom=bottoms, color=col, label=ct, width=0.5)
    for bi, (v, b) in enumerate(zip(vals, bottoms)):
        if v > 0:
            ax.text(bi, b + v / 2, str(v), ha='center', va='center',
                    fontsize=10, color='white', fontweight='bold')
    bottoms += vals
ax.set_xlabel('Distance bucket'); ax.set_ylabel('Samples')
ax.set_title('Climate coherence by distance bucket', fontsize=11)
ax.legend(title='Coherence')
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/koppen_distance.png', dpi=150, bbox_inches='tight')
plt.show()

n_coh = coherence.count('exact') + coherence.count('adjacent')
n_tot = len(coherence)
print(f'Climate-coherent: {n_coh}/{n_tot} ({100*n_coh/max(n_tot,1):.0f}%)')
print(f'Spurious (distant): {coherence.count("distant")}/{n_tot}')
print()
print('High coherent% → model confuses geographically similar places (expected for hard examples)')
print('High distant%  → model picks up spurious visual cues unrelated to geography')

## 15. Resume training after a session timeout

Re-run setup cells (sections 1–3), then run this cell to pick up from where training left off.

In [ ]:
RESUME_FROM = f'{DRIVE_DIR}/checkpoints/best.pt'

resume_trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    gallery_coords=gallery_coords,
    cfg=cfg,
    device=device,
    resume_path=RESUME_FROM,
)
print(f'Resuming from epoch {resume_trainer.start_epoch} / {cfg.training.epochs}')
resume_trainer.train()

## 16. Training curves

Reads the `metrics` dict stored in each epoch checkpoint to reconstruct the
training history. No extra logging code needed — the data is already there.

In [ ]:
import glob

ckpt_files = sorted(glob.glob(f'{DRIVE_DIR}/checkpoints/epoch_*.pt'))
if not ckpt_files:
    print('No epoch checkpoints found. Run training first.')
else:
    history = []
    for path in ckpt_files:
        ckpt = torch.load(path, map_location='cpu')
        row  = {'epoch': ckpt['epoch'] + 1}
        if 'train_loss' in ckpt:
            row['train_loss'] = ckpt['train_loss']
        row.update(ckpt.get('metrics', {}))
        history.append(row)

    epochs       = [r['epoch']       for r in history]
    mean_gcd     = [r.get('mean_gcd_km',    float('nan')) for r in history]
    median_gcd   = [r.get('median_gcd_km',  float('nan')) for r in history]
    train_losses = [r.get('train_loss',     None)         for r in history]
    thresholds   = sorted(
        [k.replace('recall@', '').replace('km', '') for k in history[0] if k.startswith('recall')],
        key=int,
    )
    recalls = {t: [r.get(f'recall@{t}km', float('nan')) * 100 for r in history]
               for t in thresholds}

    has_loss = any(v is not None for v in train_losses)
    n_plots  = 4 if has_loss else 3
    fig, axes = plt.subplots(1, n_plots, figsize=(4.5 * n_plots, 4))

    ax_idx = 0

    # Training loss (if available)
    if has_loss:
        valid = [(e, l) for e, l in zip(epochs, train_losses) if l is not None]
        axes[ax_idx].plot(*zip(*valid), marker='o', linewidth=2, color='tomato')
        axes[ax_idx].set_xlabel('Epoch'); axes[ax_idx].set_ylabel('Loss')
        axes[ax_idx].set_title('Training loss')
        axes[ax_idx].grid(alpha=0.3)
        ax_idx += 1

    # GCD over epochs
    axes[ax_idx].plot(epochs, mean_gcd,   label='Mean GCD',   marker='o', linewidth=2)
    axes[ax_idx].plot(epochs, median_gcd, label='Median GCD', marker='s', linewidth=2, linestyle='--')
    axes[ax_idx].set_xlabel('Epoch'); axes[ax_idx].set_ylabel('km')
    axes[ax_idx].set_title('Great-Circle Distance over training')
    best_epoch = epochs[median_gcd.index(min(median_gcd))]
    axes[ax_idx].axvline(best_epoch, color='red', linestyle=':', linewidth=1,
                         label=f'Best (ep {best_epoch})')
    axes[ax_idx].legend(fontsize=8); axes[ax_idx].grid(alpha=0.3)
    ax_idx += 1

    # Recall@km over epochs
    cmap = plt.cm.plasma
    for j, t in enumerate(thresholds):
        axes[ax_idx].plot(epochs, recalls[t], label=f'@{t} km', marker='o',
                          linewidth=2, color=cmap(j / max(len(thresholds) - 1, 1)))
    axes[ax_idx].set_xlabel('Epoch'); axes[ax_idx].set_ylabel('Recall (%)')
    axes[ax_idx].set_ylim(0, 105)
    axes[ax_idx].set_title('Recall@km over training')
    axes[ax_idx].legend(fontsize=7); axes[ax_idx].grid(alpha=0.3)
    ax_idx += 1

    # Final recall bar chart
    final      = history[-1]
    bar_labels = [f'@{t} km' for t in thresholds]
    bar_vals   = [final.get(f'recall@{t}km', 0) * 100 for t in thresholds]
    bar_colors = [cmap(j / max(len(thresholds) - 1, 1)) for j in range(len(thresholds))]
    bars = axes[ax_idx].bar(bar_labels, bar_vals, color=bar_colors, width=0.6,
                             edgecolor='black', linewidth=0.5)
    axes[ax_idx].set_ylabel('Recall (%)'); axes[ax_idx].set_ylim(0, 110)
    axes[ax_idx].set_title(f'Final recall — epoch {final["epoch"]}\n'
                            f'Median GCD: {final.get("median_gcd_km", 0):.0f} km')
    axes[ax_idx].grid(axis='y', alpha=0.3)
    for bar, v in zip(bars, bar_vals):
        axes[ax_idx].text(bar.get_x() + bar.get_width() / 2, v + 1.5,
                          f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

    plt.suptitle('Training history', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{DRIVE_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Best median GCD: {min(median_gcd):.0f} km  (epoch {best_epoch})')

## 17. Error distribution

Histogram and CDF of prediction errors across the whole validation set.
The log-scale x-axis shows performance across all distance regimes simultaneously —
a model that learned geography clusters mass below 200 km.

In [ ]:
# Run full-set evaluation to get per-sample distances
model.eval()
gallery_embs_full = compute_gallery_embeddings(model, gallery_coords, device=device)
all_dists = []
for imgs, coords_b in tqdm(val_loader, desc='Computing distances'):
    with torch.no_grad():
        e = model.encode_image(imgs.to(device))
        idx = (e @ gallery_embs_full.to(device).T).argmax(dim=-1)
    all_dists.append(haversine_distance(gallery_coords[idx.cpu()], coords_b))
all_dists = torch.cat(all_dists)

bins  = np.logspace(0, np.log10(20_000), 60)
thrs  = cfg.evaluation.thresholds_km

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
ax1.hist(all_dists.numpy(), bins=bins, color='steelblue', edgecolor='white', linewidth=0.3)
ax1.set_xscale('log')
ax1.set_xlabel('Error (km, log scale)'); ax1.set_ylabel('Count')
ax1.set_title('Prediction error distribution')
ax1.grid(axis='y', alpha=0.3)
for t in thrs:
    ax1.axvline(t, color='red', linewidth=1, linestyle='--', alpha=0.7)
    ax1.text(t * 1.05, ax1.get_ylim()[1] * 0.92, f'{t} km',
             color='red', fontsize=7, va='top')
ax1.axvline(all_dists.median().item(), color='orange', linewidth=1.5, linestyle='-.',
            label=f'Median: {all_dists.median():.0f} km')
ax1.legend(fontsize=8)

# CDF
sorted_d = torch.sort(all_dists).values.numpy()
cdf      = np.arange(1, len(sorted_d) + 1) / len(sorted_d) * 100
ax2.plot(sorted_d, cdf, linewidth=2, color='steelblue')
ax2.set_xscale('log')
ax2.set_xlabel('Error (km, log scale)'); ax2.set_ylabel('Cumulative % of predictions')
ax2.set_title('Cumulative error distribution (CDF)')
ax2.set_ylim(0, 105); ax2.grid(alpha=0.3)
for t in thrs:
    recall_pct = (all_dists <= t).float().mean().item() * 100
    ax2.axvline(t, color='red', linewidth=1, linestyle='--', alpha=0.7)
    ax2.axhline(recall_pct, color='red', linewidth=0.7, linestyle=':', alpha=0.5)
    ax2.scatter([t], [recall_pct], color='red', s=40, zorder=5)
    ax2.text(t * 1.08, recall_pct + 1, f'{recall_pct:.0f}%', color='red', fontsize=7)

plt.suptitle(f'Error over {len(all_dists)} validation images', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean:   {all_dists.mean():.0f} km')
print(f'Median: {all_dists.median():.0f} km')
print(f'90th p: {torch.quantile(all_dists, 0.9):.0f} km')

## 18. Embedding space (t-SNE)

Projects image and GPS embeddings into 2-D with t-SNE, coloured by Köppen group.
If the contrastive training worked, image embeddings (circles) should cluster near
their matched GPS embeddings (triangles) and the whole space should show
geographic structure — tropical points away from polar ones, etc.

In [ ]:
from sklearn.manifold import TSNE

TSNE_SAMPLES = 512   # more = slower but richer picture; cap at val set size
tsne_loader  = DataLoader(val_dataset, batch_size=64, shuffle=True)

img_embs_list, gps_embs_list, coords_list = [], [], []
model.eval()
with torch.no_grad():
    for imgs, coords_b in tsne_loader:
        ie = model.encode_image(imgs.to(device)).cpu()
        ge = model.encode_gps(coords_b.to(device)).cpu()
        img_embs_list.append(ie)
        gps_embs_list.append(ge)
        coords_list.append(coords_b)
        if sum(len(x) for x in img_embs_list) >= TSNE_SAMPLES:
            break

img_embs_t = torch.cat(img_embs_list)[:TSNE_SAMPLES]
gps_embs_t = torch.cat(gps_embs_list)[:TSNE_SAMPLES]
coords_t   = torch.cat(coords_list)[:TSNE_SAMPLES].numpy()

# Stack both modalities and reduce together so they share the same 2-D space
all_embs = torch.cat([img_embs_t, gps_embs_t]).numpy()
print(f'Running t-SNE on {len(all_embs)} embeddings (this takes ~30 s) …')
proj = TSNE(n_components=2, perplexity=40, random_state=0, n_iter=1000).fit_transform(all_embs)
img_proj = proj[:TSNE_SAMPLES]
gps_proj = proj[TSNE_SAMPLES:]

# Colour by Köppen group
from geoclip.utils.koppen import KoppenClassifier, GROUP_COLORS, KOPPEN_GROUPS
kc_tsne  = KoppenClassifier()
groups   = kc_tsne.get_group(coords_t[:, 0], coords_t[:, 1])
colors   = [GROUP_COLORS.get(g, '#aaaaaa') for g in groups]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, proj_2d, title, marker in zip(
    axes,
    [img_proj, gps_proj],
    ['Image embeddings', 'GPS embeddings'],
    ['o', '^'],
):
    sc = ax.scatter(proj_2d[:, 0], proj_2d[:, 1], c=colors, s=18,
                    marker=marker, alpha=0.75, linewidths=0)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect('equal')

patches = [mpatches.Patch(color=GROUP_COLORS[g], label=f'{g} — {KOPPEN_GROUPS[g]}')
           for g in ['A', 'B', 'C', 'D', 'E'] if g in GROUP_COLORS]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=9,
           title='Köppen group (colour = true location)')
plt.suptitle(f't-SNE of {TSNE_SAMPLES} image & GPS embeddings', fontsize=12)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig(f'{DRIVE_DIR}/tsne_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

## 19. Top-K gallery retrieval

For each query image the model ranks the entire gallery by similarity.
This view shows the top-5 retrieved GPS points on a world map alongside the image.
- A tight cluster near the true location → confident and correct.
- Candidates scattered across two continents → the model is responding to a generic
  visual cue (sky colour, compression artefacts) rather than a geographic one.

In [ ]:
TOPK     = 5
N_QUERY  = 4   # number of images to show
gallery_np_full = gallery_coords.numpy()

model.eval()
with torch.no_grad():
    q_embs = model.encode_image(images[:N_QUERY].to(device))  # reuse batch from §6
    sims_q = (q_embs @ gallery_embs.to(device).T).cpu()       # [N_QUERY, G]

topk_vals, topk_idx = sims_q.topk(TOPK, dim=-1)              # [N_QUERY, K]

fig, axes = plt.subplots(N_QUERY, 2, figsize=(13, 3.5 * N_QUERY),
                         gridspec_kw={'width_ratios': [1, 3]})

for i in range(N_QUERY):
    # Left: query image
    axes[i][0].imshow(img_nps[i])
    axes[i][0].axis('off')
    axes[i][0].set_title(
        f'Query {i}\nTrue: ({true_coords[i,0]:.1f}°, {true_coords[i,1]:.1f}°)\n'
        f'{distances[i]:.0f} km error', fontsize=8,
    )

    # Right: world map with top-K candidates
    ax = axes[i][1]
    ax.set_facecolor('#d0e8f0')
    ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
    ax.set_xticks(range(-180, 181, 60)); ax.set_yticks(range(-90, 91, 30))
    ax.tick_params(labelsize=5); ax.grid(color='white', linewidth=0.3, alpha=0.5)

    # All gallery points (small, grey)
    ax.scatter(gallery_np_full[:, 1], gallery_np_full[:, 0],
               s=1, color='#aaaaaa', alpha=0.3, linewidths=0, zorder=1)

    # Top-K candidates (larger, coloured by rank)
    cmap_k = plt.cm.YlOrRd
    for rank in range(TOPK - 1, -1, -1):   # draw highest-ranked on top
        idx_k = topk_idx[i, rank].item()
        lat_k, lon_k = gallery_np_full[idx_k]
        color_k = cmap_k(1.0 - rank / TOPK)
        ax.scatter(lon_k, lat_k, s=80 - rank * 10, color=color_k,
                   edgecolors='black', linewidths=0.5, zorder=4 + rank,
                   label=f'#{rank+1}  sim={topk_vals[i,rank]:.3f}')

    # True location
    tc = true_coords[i].numpy()
    ax.scatter(tc[1], tc[0], marker='*', s=250, color='lime',
               edgecolors='black', linewidths=0.5, zorder=10, label='True')

    ax.legend(fontsize=6, loc='lower left', framealpha=0.85)
    ax.set_title(f'Top-{TOPK} gallery candidates', fontsize=8)

# Shared colour-bar legend for ranks
sm = plt.cm.ScalarMappable(cmap=cmap_k, norm=plt.Normalize(vmin=1, vmax=TOPK))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes[:, 1], fraction=0.015, pad=0.02)
cbar.set_label('Rank (1 = highest similarity)', fontsize=8)
cbar.set_ticks(np.linspace(1, TOPK, TOPK))
cbar.set_ticklabels([f'#{r}' for r in range(1, TOPK + 1)])

plt.suptitle(f'Top-{TOPK} retrieved gallery points  (★ = true location)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/topk_retrieval.png', dpi=150, bbox_inches='tight')
plt.show()

## 20. Similarity–error calibration

Does a higher cosine similarity score actually mean a lower km error?
A well-calibrated model should show a clear negative correlation: when the model
is confident (high top-1 similarity) it should also be correct.
Weak or positive correlation means the model is overconfident on hard cases.

In [ ]:
import numpy as np

# Collect (max_sim, km_error) for every val image
model.eval()
cal_sims, cal_errs = [], []
gallery_embs_cal = compute_gallery_embeddings(model, gallery_coords, device=device)

for imgs_b, coords_b in tqdm(val_loader, desc='Calibration'):
    with torch.no_grad():
        e = model.encode_image(imgs_b.to(device))
        s = (e @ gallery_embs_cal.to(device).T).cpu()   # [B, G]
    top_sim, top_idx = s.max(dim=-1)
    pred_c = gallery_coords[top_idx]
    d = haversine_distance(pred_c, coords_b)
    cal_sims.append(top_sim)
    cal_errs.append(d)

cal_sims = torch.cat(cal_sims).numpy()
cal_errs = torch.cat(cal_errs).numpy()

# Pearson correlation
r = np.corrcoef(cal_sims, cal_errs)[0, 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Hex-bin scatter (log density)
hb = ax1.hexbin(cal_sims, cal_errs, gridsize=40, yscale='log',
                cmap='YlOrRd', mincnt=1)
plt.colorbar(hb, ax=ax1, label='Count')

# Regression line
m, b = np.polyfit(cal_sims, np.log10(cal_errs + 1), 1)
xs = np.linspace(cal_sims.min(), cal_sims.max(), 100)
ax1.plot(xs, 10 ** (m * xs + b) - 1, color='steelblue', linewidth=2, label='Trend')
ax1.set_xlabel('Max cosine similarity (confidence)')
ax1.set_ylabel('Prediction error (km, log scale)')
ax1.set_title(f'Confidence vs. error  |  r = {r:.3f}')
ax1.legend()

# Bin by similarity decile and show median error per bin
bins    = np.percentile(cal_sims, np.linspace(0, 100, 11))
bin_idx = np.digitize(cal_sims, bins[1:-1])
med_err = [np.median(cal_errs[bin_idx == i]) for i in range(10)]
pct_mid = [(bins[i] + bins[i+1]) / 2 for i in range(10)]
ax2.bar(range(10), med_err, color=plt.cm.RdYlGn(np.linspace(0, 1, 10)),
        edgecolor='black', linewidth=0.5)
ax2.set_xticks(range(10))
ax2.set_xticklabels([f'{p:.2f}' for p in pct_mid], rotation=30, fontsize=7)
ax2.set_xlabel('Mean similarity in decile')
ax2.set_ylabel('Median error (km)')
ax2.set_title('Median error per confidence decile\n(left = low confidence, right = high)')
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Similarity–error calibration', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/calibration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Pearson r (sim, error) = {r:.3f}')
if r < -0.3:
    print('→ Good calibration: high similarity reliably predicts low error.')
elif r < 0:
    print('→ Weak calibration: slight negative trend but noisy.')
else:
    print('→ Poor calibration: similarity does not predict error well — model is overconfident.')

## 21. Per-Köppen-zone performance

Breaks down recall@km and median GCD by major climate group.
Tropical (A) and polar (E) zones are visually distinctive — the model should
do well there. Dry (B) and temperate (C) zones look similar across continents
and are typically the hardest.

In [ ]:
from geoclip.training.evaluator import evaluate_by_zone
from geoclip.utils.koppen import KoppenClassifier, GROUP_COLORS, KOPPEN_GROUPS

kc_eval  = KoppenClassifier()
zone_metrics = evaluate_by_zone(
    model, val_loader, gallery_coords, kc_eval,
    device=device, thresholds_km=cfg.evaluation.thresholds_km,
)

# Print table
groups_sorted = sorted(zone_metrics, key=lambda g: -zone_metrics[g]['count'])
header = f"{'Group':>6}  {'Name':<14}  {'N':>5}  {'Median km':>10}  " + \
         '  '.join(f"R@{t}km" for t in cfg.evaluation.thresholds_km)
print(header)
print('-' * len(header))
for g in groups_sorted:
    m = zone_metrics[g]
    recalls = '  '.join(f"{m.get(f'recall@{t}km', 0)*100:>6.1f}%"
                        for t in cfg.evaluation.thresholds_km)
    name = KOPPEN_GROUPS.get(g, 'Ocean/unk')
    print(f"{g:>6}  {name:<14}  {m['count']:>5}  {m['median_gcd_km']:>10.0f}  {recalls}")

# Grouped bar chart: recall@200km and recall@2500km per zone
t1, t2 = 200, 2500
g_labels = [f"{g}\n({KOPPEN_GROUPS.get(g,'?')})" for g in groups_sorted]
r1 = [zone_metrics[g].get(f'recall@{t1}km', 0) * 100 for g in groups_sorted]
r2 = [zone_metrics[g].get(f'recall@{t2}km', 0) * 100 for g in groups_sorted]
counts = [zone_metrics[g]['count'] for g in groups_sorted]
grp_colors = [GROUP_COLORS.get(g, '#aaaaaa') for g in groups_sorted]

x = np.arange(len(groups_sorted))
w = 0.35
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

bars1 = ax1.bar(x - w/2, r1, w, label=f'Recall@{t1}km', color=grp_colors,
                alpha=0.85, edgecolor='black', linewidth=0.5)
bars2 = ax1.bar(x + w/2, r2, w, label=f'Recall@{t2}km', color=grp_colors,
                alpha=0.45, edgecolor='black', linewidth=0.5, hatch='//')
for bar, v in zip(list(bars1) + list(bars2), r1 + r2):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 0.8, f'{v:.0f}%',
             ha='center', va='bottom', fontsize=7)
ax1.set_xticks(x); ax1.set_xticklabels(g_labels, fontsize=8)
ax1.set_ylabel('Recall (%)'); ax1.set_ylim(0, 115)
ax1.set_title('Recall@km by Köppen zone')
ax1.legend(); ax1.grid(axis='y', alpha=0.3)

# Annotate sample counts
for i, n in enumerate(counts):
    ax1.text(i, -7, f'n={n}', ha='center', fontsize=7, color='grey')

# Median GCD per zone
med_gcds = [zone_metrics[g]['median_gcd_km'] for g in groups_sorted]
bars_gcd = ax2.bar(x, med_gcds, color=grp_colors, edgecolor='black', linewidth=0.5)
for bar, v in zip(bars_gcd, med_gcds):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 10, f'{v:.0f}',
             ha='center', va='bottom', fontsize=8)
ax2.set_xticks(x); ax2.set_xticklabels(g_labels, fontsize=8)
ax2.set_ylabel('Median GCD (km)')
ax2.set_title('Median error by Köppen zone')
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Per-climate-zone performance', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/zone_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 22. Ablation study

Compares three training variants to isolate the contribution of each extension.

**How to produce the three checkpoints** — run three training jobs with these config
differences (everything else identical):

| Variant | `hard_neg_swap_prob` | `lambda_attn` |
|---|---|---|
| Baseline | `0.0` | `0.0` |
| + Hard negatives | `0.5` | `0.0` |
| + Attn reg (full) | `0.5` | `0.01` |

Save each to a different sub-folder on Drive, then fill in the paths below.

In [ ]:
from geoclip.training.evaluator import evaluate
from geoclip.utils.checkpoint import load_checkpoint

# ── Fill in Drive paths for each checkpoint (skip variants you haven't trained) ──
ABLATION_CHECKPOINTS = {
    'Baseline':           f'{DRIVE_DIR}/checkpoints/baseline/best.pt',
    '+ Hard negatives':   f'{DRIVE_DIR}/checkpoints/hard_neg/best.pt',
    '+ Attn reg (full)':  f'{DRIVE_DIR}/checkpoints/full/best.pt',
}

ablation_results = {}
for name, path in ABLATION_CHECKPOINTS.items():
    if not os.path.exists(path):
        print(f'[skip] {name} — checkpoint not found at {path}')
        continue
    load_checkpoint(path, model, device=device)
    m = evaluate(model, val_loader, gallery_coords,
                 device=device, thresholds_km=cfg.evaluation.thresholds_km)
    ablation_results[name] = m
    print(f'{name}: median={m["median_gcd_km"]:.0f} km')

if len(ablation_results) < 2:
    print('\nNeed at least 2 checkpoints to compare. Run training with different configs first.')
else:
    variants = list(ablation_results)
    thrs     = cfg.evaluation.thresholds_km

    # Results table
    print(f'\n{"Variant":<25}  {"Median km":>10}  ' +
          '  '.join(f'R@{t}km' for t in thrs))
    print('-' * (25 + 12 + 9 * len(thrs)))
    for v in variants:
        m = ablation_results[v]
        row = '  '.join(f'{m.get(f"recall@{t}km",0)*100:>6.1f}%' for t in thrs)
        print(f'{v:<25}  {m["median_gcd_km"]:>10.0f}  {row}')

    # Bar chart comparison
    bar_keys   = [f'recall@{t}km' for t in thrs] + ['median_gcd_km']
    bar_titles = [f'R@{t}km (%)' for t in thrs] + ['Median GCD (km)']
    n_metrics  = len(bar_keys)
    x          = np.arange(len(variants))
    colors_ab  = ['#4472C4', '#ED7D31', '#70AD47'][:len(variants)]

    fig, axes = plt.subplots(1, n_metrics, figsize=(3.5 * n_metrics, 4.5))
    for ax, key, title in zip(axes, bar_keys, bar_titles):
        scale = 100 if 'recall' in key else 1
        vals  = [ablation_results[v].get(key, 0) * scale for v in variants]
        bars  = ax.bar(x, vals, color=colors_ab, edgecolor='black', linewidth=0.5, width=0.5)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, v * 1.02,
                    f'{v:.1f}', ha='center', va='bottom', fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels(variants, rotation=15, ha='right', fontsize=8)
        ax.set_title(title, fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        if 'recall' in key:
            ax.set_ylim(0, 115)

    plt.suptitle('Ablation study — contribution of each extension', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{DRIVE_DIR}/ablation.png', dpi=150, bbox_inches='tight')
    plt.show()